In [ ]:
Brain Stroke Prediction


In [ ]:
# ============================================================
#  BRAIN STROKE PREDICTION  |  Phase 1 — Data Loading & EDA
#  Dataset : Stroke Prediction Dataset (Kaggle – fedesoriano)
#  kaggle datasets download -d fedesoriano/stroke-prediction-dataset
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Global plot style ──────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    'figure.figsize'  : (14, 6),
    'axes.titlesize'  : 14,
    'axes.labelsize'  : 12,
    'xtick.labelsize' : 10,
    'ytick.labelsize' : 10,
})

STROKE_COLORS = {0: '#4C72B0', 1: '#DD8452'}   # No-stroke blue / Stroke orange
OUTPUT_DIR    = 'outputs'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ═══════════════════════════════════════════════════════════
#  1.  LOAD DATA
# ═══════════════════════════════════════════════════════════
print("=" * 60)
print("  BRAIN STROKE PREDICTION — Phase 1: Data Loading")
print("=" * 60)

df = pd.read_csv('healthcare-dataset-stroke-data.csv')

print(f"\n📂  Dataset loaded  →  {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\n── First 5 rows ──────────────────────────────────────")
print(df.head())


# ═══════════════════════════════════════════════════════════
#  2.  BASIC STRUCTURE
# ═══════════════════════════════════════════════════════════
print("\n\n── Data Types & Non-Null Counts ──────────────────────")
print(df.info())

print("\n── Statistical Summary (numeric) ─────────────────────")
print(df.describe().T.round(2))

print("\n── Statistical Summary (categorical) ─────────────────")
print(df.describe(include='object').T)


# ═══════════════════════════════════════════════════════════
#  3.  MISSING VALUES
# ═══════════════════════════════════════════════════════════
print("\n\n── Missing Values ────────────────────────────────────")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df  = pd.DataFrame({'Missing Count': missing,
                            'Missing %'    : missing_pct})
missing_df  = missing_df[missing_df['Missing Count'] > 0]
print(missing_df if not missing_df.empty else "✅  No missing values found.")

# Plot missing values
if not missing_df.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(x=missing_df.index, y='Missing %', data=missing_df,
                palette='Reds_r', ax=ax)
    ax.set_title('Missing Value Percentage by Column')
    ax.set_ylabel('Missing %')
    for bar in ax.patches:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.1,
                f"{bar.get_height():.2f}%",
                ha='center', va='bottom', fontsize=11)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/01_missing_values.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  → Saved: {OUTPUT_DIR}/01_missing_values.png")


# ═══════════════════════════════════════════════════════════
#  4.  DUPLICATES
# ═══════════════════════════════════════════════════════════
dups = df.duplicated().sum()
print(f"\n── Duplicate Rows ────────────────────────────────────")
print(f"  Duplicate rows found: {dups}")


# ═══════════════════════════════════════════════════════════
#  5.  TARGET DISTRIBUTION (Class Imbalance Check)
# ═══════════════════════════════════════════════════════════
print("\n── Target Distribution (stroke) ──────────────────────")
target_counts = df['stroke'].value_counts()
target_pct    = df['stroke'].value_counts(normalize=True) * 100
print(pd.DataFrame({'Count': target_counts,
                    'Percentage': target_pct.round(2)}))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
axes[0].bar(['No Stroke (0)', 'Stroke (1)'],
            target_counts.values,
            color=[STROKE_COLORS[0], STROKE_COLORS[1]], edgecolor='white', linewidth=1.2)
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 30, f"{v:,}\n({target_pct.iloc[i]:.1f}%)",
                 ha='center', fontsize=12, fontweight='bold')
axes[0].set_title('Class Distribution', fontsize=14)
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(target_counts.values,
            labels=['No Stroke', 'Stroke'],
            colors=[STROKE_COLORS[0], STROKE_COLORS[1]],
            autopct='%1.1f%%', startangle=90,
            explode=(0, 0.08), shadow=True, textprops={'fontsize': 12})
axes[1].set_title('Stroke vs No-Stroke Proportion', fontsize=14)

plt.suptitle('Target Variable Distribution — Severe Class Imbalance ⚠️', fontsize=15)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/02_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  → Saved: {OUTPUT_DIR}/02_target_distribution.png")


# ═══════════════════════════════════════════════════════════
#  6.  UNIQUE VALUES PER COLUMN
# ═══════════════════════════════════════════════════════════
print("\n── Unique Values Per Column ──────────────────────────")
for col in df.columns:
    n = df[col].nunique()
    if n <= 10:
        vals = df[col].unique().tolist()
        print(f"  {col:25s}  ({n} unique)  →  {vals}")
    else:
        print(f"  {col:25s}  ({n} unique)  →  [numeric / high-cardinality]")


print("\n\n✅  Phase 1 complete — proceed to phase2_data_cleaning.py")


In [ ]:
# ============================================================
#  BRAIN STROKE PREDICTION  |  Phase 2 — Data Cleaning
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.figsize': (14, 6), 'axes.titlesize': 14})
STROKE_COLORS = {0: '#4C72B0', 1: '#DD8452'}
OUTPUT_DIR    = 'outputs'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)


print("=" * 60)
print("  BRAIN STROKE PREDICTION — Phase 2: Data Cleaning")
print("=" * 60)


# ═══════════════════════════════════════════════════════════
#  LOAD RAW DATA
# ═══════════════════════════════════════════════════════════
df = pd.read_csv('healthcare-dataset-stroke-data.csv')
print(f"\n📂  Raw shape: {df.shape}")


# ═══════════════════════════════════════════════════════════
#  STEP 1 — Drop ID column (no predictive value)
# ═══════════════════════════════════════════════════════════
print("\n── Step 1: Drop 'id' column ──────────────────────────")
df.drop(columns=['id'], inplace=True)
print(f"  Remaining columns: {df.columns.tolist()}")


# ═══════════════════════════════════════════════════════════
#  STEP 2 — Handle 'gender' → Remove 'Other' (only 1 row)
# ═══════════════════════════════════════════════════════════
print("\n── Step 2: Gender value counts ───────────────────────")
print(df['gender'].value_counts())
other_mask = df['gender'] == 'Other'
print(f"  Rows with gender='Other': {other_mask.sum()}")
df = df[~other_mask]
print(f"  Shape after removing 'Other' gender: {df.shape}")


# ═══════════════════════════════════════════════════════════
#  STEP 3 — Handle missing BMI (3.93% null)
# ═══════════════════════════════════════════════════════════
print("\n── Step 3: Impute missing BMI ────────────────────────")
print(f"  Null BMI before: {df['bmi'].isnull().sum()}")

# Plot BMI distribution before imputation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['bmi'].dropna(), bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('BMI Distribution BEFORE Imputation')
axes[0].set_xlabel('BMI')
axes[0].set_ylabel('Count')

# Impute with median (BMI is right-skewed; median is more robust)
bmi_median = df['bmi'].median()
df['bmi'].fillna(bmi_median, inplace=True)
print(f"  Null BMI after : {df['bmi'].isnull().sum()}")
print(f"  Imputed with median BMI = {bmi_median:.2f}")

axes[1].hist(df['bmi'], bins=40, color='seagreen', edgecolor='white')
axes[1].axvline(bmi_median, color='red', linestyle='--', label=f'Median = {bmi_median:.1f}')
axes[1].set_title('BMI Distribution AFTER Median Imputation')
axes[1].set_xlabel('BMI')
axes[1].legend()

plt.suptitle('BMI — Before vs After Imputation', fontsize=14)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/03_bmi_imputation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  → Saved: {OUTPUT_DIR}/03_bmi_imputation.png")


# ═══════════════════════════════════════════════════════════
#  STEP 4 — Outlier Detection & Treatment (IQR method)
# ═══════════════════════════════════════════════════════════
print("\n── Step 4: Outlier Detection (IQR) ───────────────────")
numeric_cols = ['age', 'avg_glucose_level', 'bmi']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for idx, col in enumerate(numeric_cols):
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)].shape[0]
    print(f"  {col:22s}  Q1={Q1:.2f}  Q3={Q3:.2f}  IQR={IQR:.2f}  "
          f"Bounds=[{lower:.2f}, {upper:.2f}]  Outliers={outliers}")

    # Boxplot before
    axes[0, idx].boxplot(df[col].dropna(), vert=False, patch_artist=True,
                         boxprops=dict(facecolor='#4C72B0', alpha=0.6))
    axes[0, idx].set_title(f'{col} — Before Capping')
    axes[0, idx].set_xlabel(col)

    # Cap outliers (Winsorization)
    df[col] = df[col].clip(lower=lower, upper=upper)

    # Boxplot after
    axes[1, idx].boxplot(df[col], vert=False, patch_artist=True,
                         boxprops=dict(facecolor='#DD8452', alpha=0.6))
    axes[1, idx].set_title(f'{col} — After Capping')
    axes[1, idx].set_xlabel(col)

plt.suptitle('Outlier Treatment — IQR Winsorization', fontsize=15)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/04_outlier_treatment.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  → Saved: {OUTPUT_DIR}/04_outlier_treatment.png")


# ═══════════════════════════════════════════════════════════
#  STEP 5 — Duplicate Check
# ═══════════════════════════════════════════════════════════
print("\n── Step 5: Duplicate Check ───────────────────────────")
dups = df.duplicated().sum()
print(f"  Duplicate rows: {dups}")
if dups > 0:
    df.drop_duplicates(inplace=True)
    print(f"  Removed duplicates. New shape: {df.shape}")


# ═══════════════════════════════════════════════════════════
#  STEP 6 — Data Type Verification & Minor Fixes
# ═══════════════════════════════════════════════════════════
print("\n── Step 6: Data Types ────────────────────────────────")
# Ensure binary columns are int
df['hypertension']  = df['hypertension'].astype(int)
df['heart_disease'] = df['heart_disease'].astype(int)
df['stroke']        = df['stroke'].astype(int)
print(df.dtypes)


# ═══════════════════════════════════════════════════════════
#  STEP 7 — Save Clean DataFrame
# ═══════════════════════════════════════════════════════════
df.to_csv('stroke_clean.csv', index=False)
print(f"\n✅  Clean dataset saved → stroke_clean.csv")
print(f"   Final shape: {df.shape}")
print(f"\n── Clean Data Preview ────────────────────────────────")
print(df.head())
print(f"\n── Null Check (should be 0) ──────────────────────────")
print(df.isnull().sum())

print("\n\n✅  Phase 2 complete — proceed to phase3_eda.py")


In [ ]:
# ============================================================
#  BRAIN STROKE PREDICTION  |  Phase 3 — EDA & Analytics
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.figsize': (14, 6), 'axes.titlesize': 14})
STROKE_COLORS = {0: '#4C72B0', 1: '#DD8452'}
OUTPUT_DIR    = 'outputs'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)


print("=" * 60)
print("  BRAIN STROKE PREDICTION — Phase 3: EDA & Analytics")
print("=" * 60)


# ── Load cleaned data ──────────────────────────────────────
df = pd.read_csv('stroke_clean.csv')
print(f"\n📂  Cleaned data loaded → {df.shape}")


# ─── Helper ───────────────────────────────────────────────
def stroke_rate(grp):
    """Return stroke rate (%) within a group."""
    return grp['stroke'].mean() * 100


# ═══════════════════════════════════════════════════════════
#  3.1  NUMERIC FEATURES — Distributions split by Stroke
# ═══════════════════════════════════════════════════════════
print("\n── 3.1  Numeric Feature Distributions ───────────────")
numeric_feats = ['age', 'avg_glucose_level', 'bmi']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for idx, col in enumerate(numeric_feats):
    # KDE by stroke class
    for val, label, color in [(0, 'No Stroke', STROKE_COLORS[0]),
                               (1, 'Stroke',    STROKE_COLORS[1])]:
        sns.kdeplot(df[df['stroke'] == val][col], ax=axes[0, idx],
                    label=label, color=color, fill=True, alpha=0.4)
    axes[0, idx].set_title(f'{col} — KDE by Stroke')
    axes[0, idx].legend()

    # Violin plot
    sns.violinplot(data=df, x='stroke', y=col, ax=axes[1, idx],
                   palette=STROKE_COLORS, inner='quartile')
    axes[1, idx].set_title(f'{col} — Violin Plot')
    axes[1, idx].set_xticklabels(['No Stroke', 'Stroke'])

plt.suptitle('Numeric Features: Distribution by Stroke Outcome', fontsize=15)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/05_numeric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  → Saved: {OUTPUT_DIR}/05_numeric_distributions.png")


# ═══════════════════════════════════════════════════════════
#  3.2  CATEGORICAL FEATURES vs STROKE RATE
# ═══════════════════════════════════════════════════════════
print("\n── 3.2  Categorical Features vs Stroke Rate ──────────")
cat_feats = ['gender', 'hypertension', 'heart_disease', 'ever_married',
             'work_type', 'Residence_type', 'smoking_status']

fig, axes = plt.subplots(3, 3, figsize=(20, 16))
axes = axes.flatten()

for idx, col in enumerate(cat_feats):
    rate = df.groupby(col).apply(stroke_rate).reset_index()
    rate.columns = [col, 'stroke_rate']
    rate = rate.sort_values('stroke_rate', ascending=False)

    bars = axes[idx].bar(rate[col].astype(str), rate['stroke_rate'],
                         color='#DD8452', edgecolor='white', linewidth=1.2)
    for bar, val in zip(bars, rate['stroke_rate']):
        axes[idx].text(bar.get_x() + bar.get_width() / 2,
                       bar.get_height() + 0.1,
                       f"{val:.1f}%", ha='center', fontsize=10, fontweight='bold')
    axes[idx].set_title(f'Stroke Rate by {col}', fontsize=12)
    axes[idx].set_ylabel('Stroke Rate (%)')
    axes[idx].tick_params(axis='x', rotation=20)

# Remove extra axes
for i in range(len(cat_feats), len(axes)):
    axes[i].set_visible(False)

plt.suptitle('Stroke Rate Across Categorical Features', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/06_categorical_stroke_rates.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  → Saved: {OUTPUT_DIR}/06_categorical_stroke_rates.png")


# ═══════════════════════════════════════════════════════════
#  3.3  AGE DEEP-DIVE  (most important predictor)
# ═══════════════════════════════════════════════════════════
print("\n── 3.3  Age Deep-Dive ────────────────────────────────")
df['age_group'] = pd.cut(df['age'],
                          bins=[0, 20, 35, 50, 65, 100],
                          labels=['<20', '20-35', '35-50', '50-65', '65+'])

age_stroke = df.groupby('age_group').apply(stroke_rate).reset_index()
age_stroke.columns = ['age_group', 'stroke_rate']
age_count  = df.groupby('age_group').size().reset_index(name='count')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].bar(age_stroke['age_group'].astype(str), age_stroke['stroke_rate'],
            color=['#4C72B0', '#64A0C8', '#DD8452', '#C44E52', '#8B0000'],
            edgecolor='white')
for i, val in enumerate(age_stroke['stroke_rate']):
    axes[0].text(i, val + 0.2, f"{val:.1f}%", ha='center', fontweight='bold')
axes[0].set_title('Stroke Rate by Age Group')
axes[0].set_ylabel('Stroke Rate (%)')
axes[0].set_xlabel('Age Group')

axes[1].bar(age_count['age_group'].astype(str), age_count['count'],
            color='steelblue', edgecolor='white')
for i, val in enumerate(age_count['count']):
    axes[1].text(i, val + 10, f"{val:,}", ha='center', fontsize=10)
axes[1].set_title('Patient Count by Age Group')
axes[1].set_ylabel('Count')
axes[1].set_xlabel('Age Group')

plt.suptitle('Age Group Analysis', fontsize=15)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/07_age_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  → Saved: {OUTPUT_DIR}/07_age_analysis.png")


# ═══════════════════════════════════════════════════════════
#  3.4  GLUCOSE LEVEL DEEP-DIVE
# ═══════════════════════════════════════════════════════════
print("\n── 3.4  Glucose Level Deep-Dive ─────────────────────")
df['glucose_category'] = pd.cut(df['avg_glucose_level'],
                                  bins=[0, 100, 125, 200, 300],
                                  labels=['Normal (<100)', 'Pre-diabetic (100-125)',
                                          'Diabetic (125-200)', 'High (>200)'])

gluc_stroke = df.groupby('glucose_category').apply(stroke_rate).reset_index()
gluc_stroke.columns = ['glucose_category', 'stroke_rate']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Stacked bar count
gluc_counts = df.groupby(['glucose_category', 'stroke']).size().unstack().fillna(0)
gluc_counts.plot(kind='bar', ax=axes[0],
                 color=[STROKE_COLORS[0], STROKE_COLORS[1]],
                 edgecolor='white')
axes[0].set_title('Count by Glucose Category & Stroke')
axes[0].set_xlabel('Glucose Category')
axes[0].set_ylabel('Count')
axes[0].legend(['No Stroke', 'Stroke'])
axes[0].tick_params(axis='x', rotation=15)

# Stroke rate
axes[1].bar(gluc_stroke['glucose_category'].astype(str),
            gluc_stroke['stroke_rate'],
            color=['#64A0C8', '#F0A500', '#DD8452', '#C44E52'],
            edgecolor='white')
for i, val in enumerate(gluc_stroke['stroke_rate']):
    axes[1].text(i, val + 0.1, f"{val:.1f}%", ha='center', fontweight='bold')
axes[1].set_title('Stroke Rate by Glucose Category')
axes[1].set_ylabel('Stroke Rate (%)')
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('Average Glucose Level Analysis', fontsize=15)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/08_glucose_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  → Saved: {OUTPUT_DIR}/08_glucose_analysis.png")


# ═══════════════════════════════════════════════════════════
#  3.5  RISK FACTOR COMBINATIONS
# ═══════════════════════════════════════════════════════════
print("\n── 3.5  Compounded Risk Factors ─────────────────────")
df['risk_score'] = (
    (df['age'] > 60).astype(int) +
    df['hypertension'] +
    df['heart_disease'] +
    (df['avg_glucose_level'] > 125).astype(int) +
    (df['bmi'] > 30).astype(int)
)

risk_stroke = df.groupby('risk_score').apply(stroke_rate).reset_index()
risk_stroke.columns = ['risk_score', 'stroke_rate']
risk_count  = df.groupby('risk_score').size().reset_index(name='count')
risk_merged = risk_stroke.merge(risk_count, on='risk_score')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].bar(risk_merged['risk_score'].astype(str),
            risk_merged['stroke_rate'],
            color=plt.cm.Reds(np.linspace(0.3, 1.0, len(risk_merged))),
            edgecolor='white')
for i, row in risk_merged.iterrows():
    axes[0].text(i, row['stroke_rate'] + 0.3,
                 f"{row['stroke_rate']:.1f}%", ha='center', fontweight='bold')
axes[0].set_title('Stroke Rate by Compounded Risk Score')
axes[0].set_xlabel('Risk Score (0=low, 5=high)')
axes[0].set_ylabel('Stroke Rate (%)')

axes[1].bar(risk_merged['risk_score'].astype(str),
            risk_merged['count'],
            color='steelblue', edgecolor='white')
for i, row in risk_merged.iterrows():
    axes[1].text(i, row['count'] + 10, f"{row['count']:,}",
                 ha='center', fontsize=10)
axes[1].set_title('Patient Count by Risk Score')
axes[1].set_xlabel('Risk Score')
axes[1].set_ylabel('Count')

plt.suptitle('Compounded Risk Factor Analysis\n'
             '(Age>60 + Hypertension + Heart Disease + High Glucose + BMI>30)',
             fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/09_risk_factors.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  → Saved: {OUTPUT_DIR}/09_risk_factors.png")


# ═══════════════════════════════════════════════════════════
#  3.6  CORRELATION HEATMAP
# ═══════════════════════════════════════════════════════════
print("\n── 3.6  Correlation Heatmap (numeric) ───────────────")
# Encode categoricals for correlation
df_corr = df.copy()
le_cols = ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
for col in le_cols:
    df_corr[col] = le.fit_transform(df_corr[col].astype(str))

drop_cols = ['age_group', 'glucose_category', 'risk_score']
df_corr.drop(columns=[c for c in drop_cols if c in df_corr.columns], inplace=True)

corr = df_corr.corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax,
            annot_kws={'size': 10})
ax.set_title('Feature Correlation Heatmap', fontsize=16, pad=15)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/10_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  → Saved: {OUTPUT_DIR}/10_correlation_heatmap.png")

# Stroke correlations specifically
print("\n── Correlation with 'stroke' target ─────────────────")
stroke_corr = corr['stroke'].drop('stroke').sort_values(key=abs, ascending=False)
print(stroke_corr.round(4))


# ═══════════════════════════════════════════════════════════
#  3.7  PAIRPLOT (top features)
# ═══════════════════════════════════════════════════════════
print("\n── 3.7  Pairplot (top numeric features) ─────────────")
pair_df = df[['age', 'avg_glucose_level', 'bmi', 'stroke']].copy()
pair_df['stroke'] = pair_df['stroke'].map({0: 'No Stroke', 1: 'Stroke'})

g = sns.pairplot(pair_df, hue='stroke',
                 palette={'No Stroke': STROKE_COLORS[0], 'Stroke': STROKE_COLORS[1]},
                 plot_kws={'alpha': 0.4, 's': 20},
                 diag_kind='kde')
g.fig.suptitle('Pairplot — Age / Glucose / BMI vs Stroke', y=1.02, fontsize=14)
plt.savefig(f'{OUTPUT_DIR}/11_pairplot.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  → Saved: {OUTPUT_DIR}/11_pairplot.png")


print("\n\n✅  Phase 3 complete — proceed to phase4_model_training.py")


In [ ]:
# ============================================================
#  BRAIN STROKE PREDICTION  |  Phase 4 — Feature Engineering
#                                        & Model Training
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection      import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing        import LabelEncoder, StandardScaler
from sklearn.linear_model         import LogisticRegression
from sklearn.tree                 import DecisionTreeClassifier
from sklearn.ensemble             import (RandomForestClassifier,
                                           GradientBoostingClassifier,
                                           AdaBoostClassifier)
from sklearn.neighbors            import KNeighborsClassifier
from sklearn.svm                  import SVC
from sklearn.naive_bayes          import GaussianNB
from sklearn.metrics              import (accuracy_score, precision_score,
                                           recall_score, f1_score,
                                           roc_auc_score, confusion_matrix,
                                           classification_report, roc_curve,
                                           ConfusionMatrixDisplay)
from imblearn.over_sampling       import SMOTE
from xgboost                      import XGBClassifier
import joblib

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.figsize': (14, 6), 'axes.titlesize': 14})
STROKE_COLORS = {0: '#4C72B0', 1: '#DD8452'}
OUTPUT_DIR    = 'outputs'
MODELS_DIR    = 'saved_models'

import os
os.makedirs(OUTPUT_DIR,  exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)


print("=" * 60)
print("  BRAIN STROKE PREDICTION — Phase 4: Model Training")
print("=" * 60)


# ═══════════════════════════════════════════════════════════
#  LOAD CLEAN DATA
# ═══════════════════════════════════════════════════════════
df = pd.read_csv('stroke_clean.csv')
print(f"\n📂  Clean data loaded → {df.shape}")


# ═══════════════════════════════════════════════════════════
#  4.1  FEATURE ENGINEERING
# ═══════════════════════════════════════════════════════════
print("\n── 4.1  Feature Engineering ──────────────────────────")

df_fe = df.copy()

# 1. Age groups (ordinal)
df_fe['age_group_num'] = pd.cut(df_fe['age'],
                                  bins=[0, 20, 35, 50, 65, 100],
                                  labels=[0, 1, 2, 3, 4]).astype(int)

# 2. Glucose category (ordinal)
df_fe['glucose_cat_num'] = pd.cut(df_fe['avg_glucose_level'],
                                    bins=[0, 100, 125, 200, 300],
                                    labels=[0, 1, 2, 3]).astype(int)

# 3. BMI category (ordinal)
df_fe['bmi_cat_num'] = pd.cut(df_fe['bmi'],
                                bins=[0, 18.5, 24.9, 29.9, 100],
                                labels=[0, 1, 2, 3]).astype(int)

# 4. Compounded risk score
df_fe['risk_score'] = (
    (df_fe['age'] > 60).astype(int) +
    df_fe['hypertension'] +
    df_fe['heart_disease'] +
    (df_fe['avg_glucose_level'] > 125).astype(int) +
    (df_fe['bmi'] > 30).astype(int)
)

# 5. Age × Glucose interaction
df_fe['age_glucose'] = df_fe['age'] * df_fe['avg_glucose_level']

print(f"  New features added: age_group_num, glucose_cat_num, bmi_cat_num, "
      f"risk_score, age_glucose")


# ═══════════════════════════════════════════════════════════
#  4.2  ENCODING CATEGORICAL VARIABLES
# ═══════════════════════════════════════════════════════════
print("\n── 4.2  Encoding Categoricals ────────────────────────")

le = LabelEncoder()

# Binary encode
df_fe['gender']         = le.fit_transform(df_fe['gender'])       # F=0, M=1
df_fe['ever_married']   = le.fit_transform(df_fe['ever_married']) # N=0, Y=1
df_fe['Residence_type'] = le.fit_transform(df_fe['Residence_type'])  # R=0, U=1

# One-hot encode multi-class categoricals
df_fe = pd.get_dummies(df_fe, columns=['work_type', 'smoking_status'], drop_first=False)

print(f"  Shape after encoding: {df_fe.shape}")
print(f"  All columns:\n  {list(df_fe.columns)}")


# ═══════════════════════════════════════════════════════════
#  4.3  FEATURE / TARGET SPLIT
# ═══════════════════════════════════════════════════════════
print("\n── 4.3  Feature / Target Split ───────────────────────")
X = df_fe.drop(columns=['stroke'])
y = df_fe['stroke']
print(f"  X shape: {X.shape}   |   y shape: {y.shape}")
print(f"  Target balance:\n{y.value_counts()}")


# ═══════════════════════════════════════════════════════════
#  4.4  TRAIN / TEST SPLIT (stratified)
# ═══════════════════════════════════════════════════════════
print("\n── 4.4  Train / Test Split (80/20 stratified) ────────")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"  Train → {X_train.shape}  |  Test → {X_test.shape}")


# ═══════════════════════════════════════════════════════════
#  4.5  SMOTE — Handle Class Imbalance on TRAIN ONLY
# ═══════════════════════════════════════════════════════════
print("\n── 4.5  SMOTE Oversampling (train set) ───────────────")
print(f"  Before SMOTE: {y_train.value_counts().to_dict()}")
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f"  After  SMOTE: {pd.Series(y_train_res).value_counts().to_dict()}")

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, y_data, title in [
    (axes[0], y_train,     'BEFORE SMOTE (Train)'),
    (axes[1], y_train_res, 'AFTER  SMOTE (Train)'),
]:
    counts = pd.Series(y_data).value_counts()
    ax.bar(['No Stroke', 'Stroke'], counts.values,
           color=[STROKE_COLORS[0], STROKE_COLORS[1]], edgecolor='white')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 20, f"{v:,}", ha='center', fontweight='bold')
    ax.set_title(title)
    ax.set_ylabel('Count')

plt.suptitle('Class Balance: Before vs After SMOTE', fontsize=15)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/12_smote_balance.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  → Saved: {OUTPUT_DIR}/12_smote_balance.png")


# ═══════════════════════════════════════════════════════════
#  4.6  FEATURE SCALING
# ═══════════════════════════════════════════════════════════
print("\n── 4.6  Standard Scaling ─────────────────────────────")
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_res)
X_test_sc  = scaler.transform(X_test)

# Save scaler
joblib.dump(scaler, f'{MODELS_DIR}/scaler.pkl')
print(f"  Scaler saved → {MODELS_DIR}/scaler.pkl")


# ═══════════════════════════════════════════════════════════
#  4.7  DEFINE MODELS
# ═══════════════════════════════════════════════════════════
print("\n── 4.7  Model Definitions ────────────────────────────")

models = {
    'Logistic Regression'      : LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree'            : DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest'            : RandomForestClassifier(n_estimators=200, max_depth=8,
                                                         random_state=42, n_jobs=-1),
    'Gradient Boosting'        : GradientBoostingClassifier(n_estimators=200,
                                                             learning_rate=0.05,
                                                             max_depth=4, random_state=42),
    'XGBoost'                  : XGBClassifier(n_estimators=200, learning_rate=0.05,
                                                max_depth=5, use_label_encoder=False,
                                                eval_metric='logloss', random_state=42,
                                                n_jobs=-1),
    'AdaBoost'                 : AdaBoostClassifier(n_estimators=100, random_state=42),
    'K-Nearest Neighbors'      : KNeighborsClassifier(n_neighbors=7, n_jobs=-1),
    'Naive Bayes'              : GaussianNB(),
    'SVM (RBF)'                : SVC(kernel='rbf', probability=True, random_state=42),
}

print(f"  {len(models)} models to train: {list(models.keys())}")


# ═══════════════════════════════════════════════════════════
#  4.8  TRAIN & EVALUATE
# ═══════════════════════════════════════════════════════════
print("\n── 4.8  Training & Evaluation ────────────────────────")

results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    # Fit
    model.fit(X_train_sc, y_train_res)

    # Predict
    y_pred      = model.predict(X_test_sc)
    y_pred_prob = model.predict_proba(X_test_sc)[:, 1]

    # CV on training set
    cv_scores = cross_val_score(model, X_train_sc, y_train_res,
                                 cv=cv, scoring='roc_auc', n_jobs=-1)

    # Metrics
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    auc  = roc_auc_score(y_test, y_pred_prob)
    cv_m = cv_scores.mean()
    cv_s = cv_scores.std()

    results.append({
        'Model'         : name,
        'Accuracy'      : round(acc,  4),
        'Precision'     : round(prec, 4),
        'Recall'        : round(rec,  4),
        'F1-Score'      : round(f1,   4),
        'ROC-AUC'       : round(auc,  4),
        'CV ROC-AUC'    : round(cv_m, 4),
        'CV Std'        : round(cv_s, 4),
    })

    # Save model
    joblib.dump(model, f'{MODELS_DIR}/{name.replace(" ", "_")}.pkl')
    print(f"  ✔  {name:28s}  Acc={acc:.3f}  F1={f1:.3f}  ROC-AUC={auc:.3f}  CV={cv_m:.3f}±{cv_s:.3f}")

# Save scaler feature names
joblib.dump(X.columns.tolist(), f'{MODELS_DIR}/feature_names.pkl')


# ═══════════════════════════════════════════════════════════
#  4.9  RESULTS SUMMARY TABLE
# ═══════════════════════════════════════════════════════════
print("\n── 4.9  Summary Table ────────────────────────────────")
results_df = pd.DataFrame(results).sort_values('ROC-AUC', ascending=False)
results_df.reset_index(drop=True, inplace=True)
print(results_df.to_string(index=False))
results_df.to_csv(f'{OUTPUT_DIR}/model_comparison.csv', index=False)

# Best model by ROC-AUC
best_model_name = results_df.iloc[0]['Model']
best_model      = models[best_model_name]
print(f"\n🏆  Best model (ROC-AUC): {best_model_name}")


# ═══════════════════════════════════════════════════════════
#  4.10  METRIC COMPARISON CHARTS
# ═══════════════════════════════════════════════════════════
print("\n── 4.10 Metric Comparison Charts ─────────────────────")
metrics  = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
fig, axes = plt.subplots(len(metrics), 1, figsize=(14, 20))

for ax, metric in zip(axes, metrics):
    subset = results_df.sort_values(metric, ascending=False)
    colors = ['#C44E52' if n == best_model_name else '#4C72B0'
              for n in subset['Model']]
    bars = ax.barh(subset['Model'], subset[metric], color=colors, edgecolor='white')
    for bar, val in zip(bars, subset[metric]):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va='center', fontsize=10)
    ax.set_xlim(0, 1.08)
    ax.set_title(f'{metric} — All Models', fontsize=13)
    ax.set_xlabel(metric)

plt.suptitle('Model Comparison — All Metrics', fontsize=16)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/13_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  → Saved: {OUTPUT_DIR}/13_model_comparison.png")


# ═══════════════════════════════════════════════════════════
#  4.11  ROC CURVES — All Models
# ═══════════════════════════════════════════════════════════
print("\n── 4.11 ROC Curves ───────────────────────────────────")
fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.tab10(np.linspace(0, 1, len(models)))

for (name, model), color in zip(models.items(), colors):
    y_prob = model.predict_proba(X_test_sc)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    lw = 2.5 if name == best_model_name else 1.2
    ls = '-'  if name == best_model_name else '--'
    ax.plot(fpr, tpr, color=color, lw=lw, ls=ls,
            label=f"{name} (AUC={auc:.3f})")

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier (AUC=0.500)')
ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate',  fontsize=13)
ax.set_title('ROC Curves — All Models', fontsize=15)
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/14_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  → Saved: {OUTPUT_DIR}/14_roc_curves.png")


# ═══════════════════════════════════════════════════════════
#  4.12  CONFUSION MATRICES (Top 4 Models)
# ═══════════════════════════════════════════════════════════
print("\n── 4.12 Confusion Matrices (Top 4) ───────────────────")
top4  = results_df.head(4)['Model'].tolist()
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes  = axes.flatten()

for idx, name in enumerate(top4):
    model  = models[name]
    y_pred = model.predict(X_test_sc)
    cm     = confusion_matrix(y_test, y_pred)
    disp   = ConfusionMatrixDisplay(confusion_matrix=cm,
                                    display_labels=['No Stroke', 'Stroke'])
    disp.plot(ax=axes[idx], colorbar=False, cmap='Blues')
    axes[idx].set_title(f'{name}', fontsize=12)
    axes[idx].grid(False)

plt.suptitle('Confusion Matrices — Top 4 Models', fontsize=15)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/15_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  → Saved: {OUTPUT_DIR}/15_confusion_matrices.png")


# ═══════════════════════════════════════════════════════════
#  4.13  FEATURE IMPORTANCE (Best Tree-Based Model)
# ═══════════════════════════════════════════════════════════
print("\n── 4.13 Feature Importance ───────────────────────────")
# Use Random Forest for stable importances
rf_model = models['Random Forest']
feat_imp  = pd.Series(rf_model.feature_importances_,
                       index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.RdYlGn_r(np.linspace(0, 1, len(feat_imp)))
feat_imp.plot(kind='barh', ax=ax, color=colors[::-1], edgecolor='white')
ax.set_title('Feature Importance — Random Forest', fontsize=15)
ax.set_xlabel('Importance Score')
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/16_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"  → Saved: {OUTPUT_DIR}/16_feature_importance.png")

print("\nTop 10 Features:")
print(feat_imp.head(10).round(4))


# ═══════════════════════════════════════════════════════════
#  4.14  DETAILED REPORT — Best Model
# ═══════════════════════════════════════════════════════════
print(f"\n── 4.14 Classification Report ({best_model_name}) ───")
best_preds = best_model.predict(X_test_sc)
print(classification_report(y_test, best_preds,
                             target_names=['No Stroke', 'Stroke']))


print("\n\n✅  Phase 4 complete — all models saved to saved_models/")
print(f"   Best model: {best_model_name}")
print("   Proceed to phase5_prediction.py for inference pipeline.")


In [ ]:
# ============================================================
#  BRAIN STROKE PREDICTION  |  Phase 5 — Inference Pipeline
#  Use trained models to predict stroke risk for new patients
# ============================================================

import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings('ignore')

MODELS_DIR = 'saved_models'


# ═══════════════════════════════════════════════════════════
#  LOAD ARTIFACTS
# ═══════════════════════════════════════════════════════════
scaler        = joblib.load(f'{MODELS_DIR}/scaler.pkl')
feature_names = joblib.load(f'{MODELS_DIR}/feature_names.pkl')

# Load best models for ensemble / individual predictions
models = {
    'Random Forest'     : joblib.load(f'{MODELS_DIR}/Random_Forest.pkl'),
    'XGBoost'           : joblib.load(f'{MODELS_DIR}/XGBoost.pkl'),
    'Gradient Boosting' : joblib.load(f'{MODELS_DIR}/Gradient_Boosting.pkl'),
}


# ═══════════════════════════════════════════════════════════
#  PREPROCESSING FUNCTION  (mirrors Phase 4 exactly)
# ═══════════════════════════════════════════════════════════
def preprocess_patient(patient_dict: dict) -> np.ndarray:
    """
    Takes a raw patient dict and returns a scaled feature vector
    ready for model inference.

    Parameters
    ----------
    patient_dict : dict  with keys:
        gender          : 'Male' | 'Female'
        age             : float
        hypertension    : 0 | 1
        heart_disease   : 0 | 1
        ever_married    : 'Yes' | 'No'
        work_type       : 'Private' | 'Self-employed' | 'Govt_job'
                          | 'children' | 'Never_worked'
        Residence_type  : 'Urban' | 'Rural'
        avg_glucose_level : float
        bmi             : float
        smoking_status  : 'formerly smoked' | 'never smoked'
                          | 'smokes' | 'Unknown'

    Returns
    -------
    np.ndarray : scaled (1, n_features) array
    """
    p = patient_dict.copy()

    # ── Binary encodings ──────────────────────────────────
    p['gender']         = 1 if p['gender'] == 'Male'  else 0
    p['ever_married']   = 1 if p['ever_married'] == 'Yes' else 0
    p['Residence_type'] = 1 if p['Residence_type'] == 'Urban' else 0

    # ── Derived / interaction features ───────────────────
    p['age_group_num']    = (0 if p['age'] <= 20 else
                             1 if p['age'] <= 35 else
                             2 if p['age'] <= 50 else
                             3 if p['age'] <= 65 else 4)
    p['glucose_cat_num']  = (0 if p['avg_glucose_level'] <= 100 else
                             1 if p['avg_glucose_level'] <= 125 else
                             2 if p['avg_glucose_level'] <= 200 else 3)
    p['bmi_cat_num']      = (0 if p['bmi'] < 18.5 else
                             1 if p['bmi'] < 25.0 else
                             2 if p['bmi'] < 30.0 else 3)
    p['risk_score']       = (int(p['age'] > 60) + p['hypertension'] +
                             p['heart_disease'] +
                             int(p['avg_glucose_level'] > 125) +
                             int(p['bmi'] > 30))
    p['age_glucose']      = p['age'] * p['avg_glucose_level']

    # ── One-hot: work_type ────────────────────────────────
    work_types = ['Govt_job', 'Never_worked', 'Private',
                  'Self-employed', 'children']
    for wt in work_types:
        p[f'work_type_{wt}'] = 1 if p.get('work_type') == wt else 0

    # ── One-hot: smoking_status ───────────────────────────
    smoke_cats = ['Unknown', 'formerly smoked', 'never smoked', 'smokes']
    for sc in smoke_cats:
        p[f'smoking_status_{sc}'] = 1 if p.get('smoking_status') == sc else 0

    # ── Drop raw categorical fields ───────────────────────
    for key in ['work_type', 'smoking_status']:
        p.pop(key, None)

    # ── Build ordered feature vector ─────────────────────
    row = pd.DataFrame([p], columns=feature_names).fillna(0)
    return scaler.transform(row)


# ═══════════════════════════════════════════════════════════
#  PREDICT FUNCTION
# ═══════════════════════════════════════════════════════════
def predict_stroke_risk(patient_dict: dict,
                         threshold: float = 0.35) -> dict:
    """
    Returns stroke probability and risk level for a patient.

    Parameters
    ----------
    patient_dict : dict  (see preprocess_patient for keys)
    threshold    : float — decision boundary (default 0.35 for higher recall)

    Returns
    -------
    dict with keys: probabilities, ensemble_prob, prediction, risk_level
    """
    X_scaled = preprocess_patient(patient_dict)
    probs    = {}
    for name, model in models.items():
        probs[name] = round(model.predict_proba(X_scaled)[0][1], 4)

    ensemble_prob = round(np.mean(list(probs.values())), 4)
    prediction    = int(ensemble_prob >= threshold)

    if ensemble_prob < 0.20:
        risk_level = '🟢 LOW RISK'
    elif ensemble_prob < 0.40:
        risk_level = '🟡 MODERATE RISK'
    elif ensemble_prob < 0.60:
        risk_level = '🟠 HIGH RISK'
    else:
        risk_level = '🔴 VERY HIGH RISK'

    return {
        'individual_probs' : probs,
        'ensemble_prob'    : ensemble_prob,
        'prediction'       : prediction,
        'risk_level'       : risk_level,
    }


# ═══════════════════════════════════════════════════════════
#  DEMO PREDICTIONS
# ═══════════════════════════════════════════════════════════
print("=" * 60)
print("  BRAIN STROKE PREDICTION — Phase 5: Inference Demo")
print("=" * 60)

test_cases = [
    # Case 1 – Low risk: young, healthy
    {
        'label'             : 'Case 1 — Young Healthy Male',
        'gender'            : 'Male',
        'age'               : 27.0,
        'hypertension'      : 0,
        'heart_disease'     : 0,
        'ever_married'      : 'No',
        'work_type'         : 'Private',
        'Residence_type'    : 'Urban',
        'avg_glucose_level' : 88.0,
        'bmi'               : 22.5,
        'smoking_status'    : 'never smoked',
    },
    # Case 2 – Moderate risk: middle-aged, hypertensive
    {
        'label'             : 'Case 2 — Middle-aged Hypertensive Female',
        'gender'            : 'Female',
        'age'               : 52.0,
        'hypertension'      : 1,
        'heart_disease'     : 0,
        'ever_married'      : 'Yes',
        'work_type'         : 'Self-employed',
        'Residence_type'    : 'Rural',
        'avg_glucose_level' : 118.0,
        'bmi'               : 28.4,
        'smoking_status'    : 'formerly smoked',
    },
    # Case 3 – High risk: elderly, multiple comorbidities
    {
        'label'             : 'Case 3 — Elderly High-Risk Male',
        'gender'            : 'Male',
        'age'               : 72.0,
        'hypertension'      : 1,
        'heart_disease'     : 1,
        'ever_married'      : 'Yes',
        'work_type'         : 'Private',
        'Residence_type'    : 'Urban',
        'avg_glucose_level' : 195.0,
        'bmi'               : 34.2,
        'smoking_status'    : 'smokes',
    },
]

print()
for case in test_cases:
    label  = case.pop('label')
    result = predict_stroke_risk(case, threshold=0.35)
    print(f"─── {label} ───")
    for model_name, prob in result['individual_probs'].items():
        print(f"  {model_name:22s}: {prob:.2%}")
    print(f"  {'Ensemble Probability':22s}: {result['ensemble_prob']:.2%}")
    print(f"  {'Prediction':22s}: {'STROKE' if result['prediction'] == 1 else 'NO STROKE'}")
    print(f"  {'Risk Level':22s}: {result['risk_level']}")
    print()


# ═══════════════════════════════════════════════════════════
#  BATCH PREDICTION FROM CSV
# ═══════════════════════════════════════════════════════════
def batch_predict(input_csv: str, output_csv: str,
                   threshold: float = 0.35) -> pd.DataFrame:
    """
    Run predictions on a batch CSV (same schema as training data, no 'stroke' col).
    Appends columns: stroke_probability, stroke_prediction, risk_level.
    """
    df      = pd.read_csv(input_csv)
    results = []
    for _, row in df.iterrows():
        patient = row.to_dict()
        patient.pop('id', None)
        patient.pop('stroke', None)
        res = predict_stroke_risk(patient, threshold)
        results.append({
            'ensemble_prob' : res['ensemble_prob'],
            'prediction'    : res['prediction'],
            'risk_level'    : res['risk_level'],
        })
    out_df = pd.concat([df, pd.DataFrame(results)], axis=1)
    out_df.to_csv(output_csv, index=False)
    print(f"\n✅  Batch predictions saved → {output_csv}")
    return out_df


print("\n✅  Phase 5 complete.")
print("   Call predict_stroke_risk(patient_dict) for single inference.")
print("   Call batch_predict(input_csv, output_csv) for bulk inference.")
